# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
mdict = metadata.to_json()

print(f"{mdict['name']}: {mdict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSet @ids and their fields with their @ids
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}, @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name}, @id: {field.id}, DataType: {field.data_type}")
    print()

# For convenience, store the first record set id (assuming main table is first)
if record_sets:
    main_record_set_id = record_sets[0].id
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set, indexed by RecordSet @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for RecordSet {record_set_id}")

print(f"Loaded record sets: {list(dataframes.keys())}\n")
if main_record_set_id:
    print(f"Columns in main RecordSet ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field for analysis: find first float/integer field in main record set
import numpy as np

numeric_field_id = None
group_field_id = None
if main_record_set_id:
    rs = [x for x in record_sets if x.id == main_record_set_id][0]
    for field in rs.fields:
        if field.data_type in ['schema:Float', 'schema:Integer', 'float', 'integer', 'number', 'Float', 'Integer', 'Number']:
            numeric_field_id = field.id
            break
    # Choose a grouping/categorical field as example (first string or other non-numeric)
    for field in rs.fields:
        if field.data_type in ['schema:Text', 'text', 'string', 'Text'] and field.id != numeric_field_id:
            group_field_id = field.id
            break
    df = dataframes[main_record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        # Coerce to numeric in case
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value): Total = {len(filtered_df)}")

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (showing first 5):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}: (showing first 5)")
            display(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("No main record set available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Cannot visualize: No valid numeric field loaded.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

> In this notebook, we loaded the FAIR^2 Mass Spectrometry dataset using the provided Croissant schema. We listed record sets and fields by their `@id`, extracted data using `mlcroissant`, and ran basic filtering, normalization, grouping, and visualization steps. The structure and richness of the dataset enable custom downstream analyses, including protein expression patterns and modifications specific to human extracellular vesicles. Further domain-specific interpretation is possible by referencing field `@id`s and consulting dataset documentation.